## Robustness and Comparative Analysis

This notebook implements a robustness check for the counterfactual framework applied to the Fuel Tank Filter case study: a **SHAP-naive baseline**, quantifying the added value of the counterfactual module over a naive decision rule derived directly from SHAP feature importance (Santos et al.'s Triad+SHAP method).

Uses the original surrogate model (`../models/gbc_best_model.pkl`). Requires scikit-learn `1.4.1.post1` exactly (pinned in `requirements.txt`), as the model was pickled with this version and newer scikit-learn releases (>=1.6) changed the internal `_loss` module in a way that breaks unpickling.

In [1]:
import numpy as np
import pandas as pd
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler

my_seed = 42
np.random.seed(my_seed)

/Users/mariapereda/Library/CloudStorage/Dropbox/UPM/investigacion/Josema&cia/Counterfactuals/Repo_comun/counterEVM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load data (same procedure as `cfs_generation.ipynb`)

In [2]:
data = pd.read_csv('../sim_data/C201905_counterfactuals/simulation_C201905_counterfactuals__ev0.1812.csv', index_col=0)
data['delay'] = (data['actual_duration'] > data['baseline_duration']) * 1
data.drop(columns=['actual_duration', 'baseline_duration', 'critical_path'], inplace=True)
columns_sorted = sorted(data.columns, key=lambda x: int(x.replace('duration', '')) if 'duration' in x else float('inf'))
data = data[columns_sorted]

def normalize_dataframe(df, feature_range=(0, 1)):
    scaler = MinMaxScaler(feature_range=feature_range)
    normalized_data = scaler.fit_transform(df)
    return pd.DataFrame(normalized_data, columns=df.columns, index=df.index), scaler

def denormalize_dataframe(normalized_df, scaler):
    denormalized_data = scaler.inverse_transform(normalized_df)
    return pd.DataFrame(denormalized_data, columns=normalized_df.columns, index=normalized_df.index)

data_to_normalize = data.iloc[:, :-1]
normalized_data, my_scaler = normalize_dataframe(data_to_normalize)
data = pd.concat([normalized_data, data.iloc[:, -1]], axis=1)

X = data.drop(columns=['delay'])
y = data['delay']

query_instance = X.loc[[11030]]
class_of_query_instance = y.loc[[11030]]
print('Query instance class (1=delay):', class_of_query_instance.values)

Query instance class (1=delay): [1]


### Load the original surrogate model

In [3]:
gbc_model = joblib.load('../models/gbc_best_model.pkl')
print('Prediction on query instance:', gbc_model.predict(query_instance))

Prediction on query instance: [1]


### SHAP analysis of the query instance

Follows the same approach as `sheva_shapley_backward_analysis.ipynb` (Santos et al.'s Triad+SHAP method): instance-level SHAP values via `shap.Explainer`, ranking tasks by their contribution towards the delayed-completion class.

In [4]:
explainer = shap.Explainer(gbc_model, X)
shap_values = explainer(query_instance)

vals = shap_values.values
if vals.ndim == 3:
    vals = vals[:, :, 1]
vals = vals[0]

shap_ranking = pd.Series(vals, index=X.columns).sort_values(ascending=False)
print('SHAP ranking towards delay (highest first):')
print(shap_ranking)

SHAP ranking towards delay (highest first):
duration11    2.969695
duration2     1.109163
duration12    0.135179
duration15    0.015280
duration6     0.007095
duration7     0.005535
duration3     0.002244
duration4     0.002231
duration8     0.000605
duration5    -0.001007
duration9    -0.004275
duration1    -0.006957
duration13   -0.022527
duration10   -0.025067
duration14   -0.349038
dtype: float64


### Naive SHAP-driven baseline

Simulates the decision a manager would make using only the SHAP importance ranking, without the counterfactual search machinery: compress the highest-ranked task progressively towards its optimistic bound ($0.8 \times$ baseline duration) until the surrogate predicts on-time completion. If a single task is not enough, escalate to the next-ranked task, and so on.

The cost function is identical to Equation 1 in the manuscript (`calculate_project_cost` in `cfs_generation.ipynb`).

In [5]:
def calculate_project_cost(instance_duration, base_duration, linear_cost, alpha, beta):
    d = instance_duration.squeeze()
    d_base = base_duration.squeeze()
    c = linear_cost.squeeze()
    d_optimistic = d_base * 0.8
    d_pessimistic = d_base * 1.2
    cost = pd.Series(index=d.index, dtype=float)
    for task in d.index:
        if d[task] < d_base[task]:
            denominator = (d_base[task] - d_optimistic[task]) ** 2
            adjustment = 1 + alpha * (d_base[task] - d[task]) / denominator if denominator != 0 else 1
        else:
            denominator = (d_pessimistic[task] - d_base[task]) ** 2
            adjustment = 1 + beta * (d[task] - d_base[task]) / denominator if denominator != 0 else 1
        cost[task] = d[task] * c[task] * adjustment
    return cost.sum()

costs = [122.77, 151.32, 291.67, 67.88, 2.82, 49.07, 75.00, 6.19, 37.45, 7.90, 9.21, 1554.69, 0.00, 921.30, 0.00]
df_costs = pd.DataFrame([costs], columns=[f"duration{i+1}" for i in range(15)])
base_durations_days = [224, 209, 30, 267, 222, 242, 250, 202, 247, 269, 258, 16, 1, 27, 1]
base_durations_hours = pd.DataFrame([base_durations_days], columns=[f"duration{i+1}" for i in range(15)]) * 8
alpha, beta = 0.1, 0

In [6]:
shap_task_order = [t for t in shap_ranking.index if shap_ranking[t] > 0]
print('Tasks pushing towards delay, in SHAP order:', shap_task_order)

denorm_query = denormalize_dataframe(query_instance, my_scaler)
current_duration = denorm_query.copy()
d_base = base_durations_hours.squeeze()
d_optimistic = d_base * 0.8

flipped = False
tasks_touched = []
STEPS = 50

feature_list = list(X.columns)
for task in shap_task_order:
    tasks_touched.append(task)
    lo, hi = d_optimistic[task], current_duration[task].values[0]
    for step in range(1, STEPS + 1):
        trial_value = hi - (hi - lo) * step / STEPS
        trial_duration = current_duration.copy()
        trial_duration[task] = trial_value

        trial_norm = trial_duration.copy()
        for col in trial_norm.columns:
            idx = feature_list.index(col)
            col_min = my_scaler.data_min_[idx]
            col_max = my_scaler.data_max_[idx]
            trial_norm[col] = (trial_duration[col] - col_min) / (col_max - col_min)

        pred = gbc_model.predict(trial_norm[feature_list])[0]
        if pred == 0:
            flipped = True
            break
    if flipped:
        current_duration[task] = trial_value
        break
    else:
        current_duration[task] = lo

print('Flipped to on-time after touching task(s):', tasks_touched if flipped else 'DID NOT FLIP')
naive_cost = calculate_project_cost(current_duration, base_durations_hours, df_costs, alpha, beta)
print('Naive SHAP-baseline cost: {:.2f}'.format(naive_cost))

Tasks pushing towards delay, in SHAP order: ['duration11', 'duration2', 'duration12', 'duration15', 'duration6', 'duration7', 'duration3', 'duration4', 'duration8']
Flipped to on-time after touching task(s): ['duration11', 'duration2']
Naive SHAP-baseline cost: 1511669.93


### Comparison with the best published counterfactual (CF1)

In [7]:
cf1_cost = 1467445.334426318  # from ../sim_data/comparison_df_sorted.csv, row 0 (lowest-cost CF)
pct_diff = (naive_cost - cf1_cost) / cf1_cost * 100
print('CF1 (best actual counterfactual) cost: {:.2f}'.format(cf1_cost))
print('Naive SHAP-baseline cost:              {:.2f}'.format(naive_cost))
print('Naive baseline is {:.2f}% {} than CF1'.format(abs(pct_diff), 'more expensive' if pct_diff > 0 else 'cheaper'))

CF1 (best actual counterfactual) cost: 1467445.33
Naive SHAP-baseline cost:              1511669.93
Naive baseline is 3.01% more expensive than CF1
